# Benchmark Methodology — Nondeterminism, Significance, and Honest Comparison

[Trajectory analysis](trajectory_analysis.ipynb) and
[evaluation metrics](../vo_evaluation_metrics.ipynb) answer *what to measure*.
[Estimator consistency](estimator_consistency.ipynb) adds *whether the
uncertainty is honest*. This notebook is about the layer above all three: given
a metric, **how do you run it so the resulting comparison means something?**

The typical VO/SLAM results table looks like this:

| Method | MH01 | MH02 | MH03 | MH04 | MH05 |
|---|---|---|---|---|---|
| Ours | **0.13** | **0.16** | **0.39** | **0.68** | **0.52** |
| Baseline | 0.15 | 0.18 | 0.42 | 0.71 | 0.55 |

Three digits of precision, no error bars, one run per cell, and every number
bolded in our favour. Nothing in that table says whether a re-run would produce
it again. This notebook covers what it would take to make it defensible:
where run-to-run variance comes from and how large it is (§1–2), the right
statistical test (§3–4), the aggregation choices that silently change the
ranking (§5), and the reporting checklist (§7).

**Companion docs:**

* [Trajectory analysis](trajectory_analysis.ipynb) — the metrics being aggregated here
* [Estimator consistency](estimator_consistency.ipynb) — the third evaluation axis; §2 here supplies its Monte-Carlo runs
* [Runtime evaluation](runtime_evaluation.ipynb) — the same rigour applied to timing
* [Head-to-head VIO comparison](../../vio_benchmark/docs/COMPARISON.md) — this repo's own results, which the checklist in §7 applies to


## 1. "Deterministic" algorithms are not

The reflex objection is that VO is deterministic: same input, same output, so
one run is enough. That is false for essentially every real system, for at
least six independent reasons.

| Source | Mechanism |
|---|---|
| **RANSAC / PROSAC** | Randomised sampling. Different seed → different inlier set → different pose → different keyframe decision → divergent trajectory. This alone dominates in feature-based systems. |
| **Thread scheduling** | Floating-point addition is not associative. A parallel reduction (OpenMP, TBB, `std::reduce`) sums in whatever order threads finish, so the *bitwise* result changes run to run. Tiny at first, then amplified by every nonlinear decision downstream. |
| **Real-time frame dropping** | This is the big one for ROS systems. Run in real time and the scheduler decides which frames get dropped under load. A different drop pattern is a *different input sequence*. |
| **GPU nondeterminism** | Atomics and reduction order in CUDA kernels; cuDNN algorithm auto-tuning picks different kernels depending on transient VRAM state. |
| **Timing-dependent logic** | Keyframe selection, marginalisation triggers, and loop-closure attempts that depend on wall clock or on queue depth rather than on frame index. |
| **Initialisation** | Which frames the system chooses to initialise on, and whether initialisation succeeds at all, often depends on all of the above. |

The consequence: **a benchmark number is a sample from a distribution**, and
reporting one sample without its spread is reporting an unknown quantity.


### 1.1 Worked example — the spread, and the cost of reporting the best run

Ten runs of the same system, same sequence, same parameters, different seeds.
ATE in metres:

$$
\{0.42,\ 0.39,\ 0.51,\ 0.44,\ 0.40,\ 0.88,\ 0.43,\ 0.41,\ 0.47,\ 0.45\}
$$

**Mean.** The sum is $4.80$, so $\bar{x} = 4.80/10 = 0.480$ m.

**Median.** Sorted:
$\{0.39, 0.40, 0.41, 0.42, \underline{0.43}, \underline{0.44}, 0.45, 0.47, 0.51, 0.88\}$,
so the median is $(0.43 + 0.44)/2 = 0.435$ m.

**Standard deviation.** Deviations from $0.480$:
$\{-0.06, -0.09, 0.03, -0.04, -0.08, 0.40, -0.05, -0.07, -0.01, -0.03\}$,
whose squares sum to $0.189$. With $M-1 = 9$:

$$
s = \sqrt{0.189/9} = \sqrt{0.021} = 0.145\ \text{m},
\qquad
\text{SEM} = \frac{0.145}{\sqrt{10}} = 0.046\ \text{m}
$$

So the honest single-line summary is $\mathbf{0.48 \pm 0.15}$ m (or
$0.48 \pm 0.05$ m if you mean the standard error of the mean — **say which**).

**Now the three ways to report this run set:**

| Reported | Value | What it is |
|---|---|---|
| "0.39 m" | min of 10 | the best of ten draws |
| "0.435 m" | median | the typical run |
| "0.48 ± 0.15 m" | mean ± sd | the actual answer |

Reporting the minimum buys a **12 % improvement over the median with zero
algorithmic change**, and 19 % over the mean. That is larger than the margin in
most published comparison tables. It is also not necessarily deliberate: run
the pipeline a few times while debugging, keep the run that looked best, and
you have done min-of-$N$ selection without noticing.

**And the competitor.** A rival method reporting a single run at 0.46 m is
**statistically indistinguishable** from this one — 0.46 sits comfortably inside
$[0.39, 0.88]$. Any claim of superiority from those two numbers is unsupported.

Note also the 0.88 outlier. One run in ten went noticeably wrong. Whether that
is a near-failure that the mean should absorb, or a distinct failure mode that
deserves its own row, is a judgement call — but it must be *visible*, which
means reporting max or a boxplot, not just the mean. Compare
[trajectory analysis §6](trajectory_analysis.ipynb), which makes the same
argument about frames within one run.


## 2. How many runs, and what to do with them

**For accuracy:** 5–10 runs per (algorithm, sequence) is enough to expose the
spread and to make min-of-$N$ selection impossible to hide. Report median and
IQR, or mean and sd, plus the number of runs.

**For consistency (NEES/ANEES):** many more — see
[estimator consistency §4](estimator_consistency.ipynb), where $M = 50$ is what
it takes to get the acceptance interval down to a factor of 1.75. Note that
seed-resampled runs are *not* the independent noise realisations that ANEES
strictly requires (§4.1(b) there); they are a lower bound on the spread, not a
valid $\chi^2$ sample.

**Tooling.** `rpg_trajectory_evaluation`'s `analyze_trajectories.py` supports a
matrix of algorithms × sequences × **runs** natively, and produces the
per-algorithm boxplots and the mean/median/min/max/std LaTeX tables from it.
The multiple-runs dimension is the least-used feature of the most-used
evaluation toolbox in the field. `evo` has no multi-run notion at all — you
aggregate its `.zip` results yourself, or use `evo_res` on several at once.

**A cheap discipline that catches most problems:** fix the seed, run twice,
diff the trajectories. If they are not bit-identical, you have nondeterminism
and you need the run distribution. If they *are* identical, you have only
established determinism under that seed — vary the seed and repeat.


## 3. Compare paired, not unpaired

When comparing two methods across sequences, the dominant source of variance is
**sequence difficulty**, not method quality. MH04 is hard for everyone; MH01 is
easy for everyone. An unpaired test treats that shared difficulty as noise and
buries the effect you care about.

Use the **paired** design: same sequences, same seeds where the code allows it,
and test the *differences*.

### 3.1 Worked example — the same data, $t = 0.17$ or $t = 10.6$

ATE in metres, one run each (for clarity — in practice combine this with §2):

| Sequence | A | B | $d = A - B$ |
|---|---|---|---|
| MH01 | 0.15 | 0.13 | 0.02 |
| MH02 | 0.18 | 0.16 | 0.02 |
| MH03 | 0.42 | 0.39 | 0.03 |
| MH04 | 0.71 | 0.68 | 0.03 |
| MH05 | 0.55 | 0.52 | 0.03 |

Means: $\bar{A} = 2.01/5 = 0.402$, $\bar{B} = 1.88/5 = 0.376$. B is better by
$0.026$ m either way — the *estimate* is not in dispute. The question is
whether it is distinguishable from noise.

**Unpaired two-sample $t$-test.** Each sample's standard deviation is
dominated by the sequence spread: $s_A = 0.2397$, $s_B = 0.2348$, so the pooled
variance is $s_p^2 = (0.2397^2 + 0.2348^2)/2 = 0.05629$. Then

$$
t = \frac{0.402 - 0.376}{\sqrt{s_p^2\left(\frac15 + \frac15\right)}}
  = \frac{0.026}{\sqrt{0.05629 \times 0.4}} = \frac{0.026}{0.1501} = \mathbf{0.173},
\qquad \text{df} = 8,\quad p = 0.87
$$

Not remotely significant. The between-sequence spread (0.15 m to 0.71 m)
swamps a 0.026 m effect.

**Paired $t$-test.** Work with the five differences
$\{0.02, 0.02, 0.03, 0.03, 0.03\}$:

$$
\bar{d} = \frac{0.13}{5} = 0.026, \qquad
s_d = \sqrt{\frac{0.00012}{4}} = \sqrt{3\times10^{-5}} = 0.005477
$$

$$
t = \frac{\bar{d}}{s_d/\sqrt{5}} = \frac{0.026}{0.005477/2.2361}
  = \frac{0.026}{0.002449} = \mathbf{10.61},
\qquad \text{df} = 4,\quad p = 0.00045
$$

against the critical value $t_{0.975,\,4} = 2.776$. Strongly significant.

**Same five pairs of numbers, $t$ from 0.17 to 10.6, purely from pairing.** The
sequence difficulty is a nuisance factor that pairing removes exactly. Running
the unpaired test on VO benchmark data is not conservative — it is the wrong
test, and it will tell you that a real, reproducible improvement is noise.

**Caveat that matters more than the $p$-value:** the improvement is 0.026 m on
errors of 0.13–0.71 m, i.e. **6 %**. Highly significant and probably
operationally irrelevant. Report the effect size and its confidence interval;
the $p$-value alone says only that the effect is not zero.


## 4. Confidence intervals without the normality assumption

With 5 sequences, the $t$-test's normality assumption is doing real work and
cannot be checked. The **percentile bootstrap** avoids it:

1. Resample the $N$ sequences **with replacement** to get a new set of size $N$.
2. Recompute the aggregate statistic (mean ATE, median ATE, mean paired
   difference — whatever you report).
3. Repeat $B = 10{,}000$ times.
4. The 2.5th and 97.5th percentiles of the $B$ values are the 95 % CI.

**The resampling unit must be the unit of independence.** For a benchmark that
is the **sequence** (or the (sequence, run) pair), never the frame. Resampling
frames within a trajectory treats 3000 heavily-correlated poses as 3000
independent samples and produces a CI roughly $\sqrt{K_{\text{eff}}/K}$ times
too narrow — the same effective-sample-size error as time-averaged NEES in
[estimator consistency §4.1(d)](estimator_consistency.ipynb). This is a common
mistake in per-frame APE error bars, and it makes any two methods look
significantly different.

For the paired design, bootstrap the **differences**, which keeps the pairing
intact and gives a CI on the effect size directly.

With $N = 5$ sequences the bootstrap is honest but weak — there are only $5^5 =
3125$ distinct resamples, and the CI will be wide. That is the correct
conclusion, not a defect of the method: five sequences cannot support a
confident claim about a 6 % effect. The fix is more sequences, not a different
test.


## 5. Aggregation traps — where the ranking silently changes

### 5.1 Failures, and the biggest silent cheat in the field

What do you do with a run that loses tracking, diverges, or never initialises?

**Worked example.**

| System | Completed | Mean ATE over completed runs |
|---|---|---|
| A | 10 / 10 | 0.50 m |
| B | 6 / 10 | 0.30 m |

The table that gets published — "A: 0.50, **B: 0.30**" — makes B the winner by
40 %. But B fails 40 % of the time, and it fails on the hard parts, which is
exactly why its average over the surviving runs is low: **the exclusion is
correlated with difficulty.** For any real deployment A is strictly better.

Rules:

* **Always report completion / success rate as its own column.** It is not a
  footnote.
* Never average over only the successful runs without saying so, in the caption,
  next to the number.
* Consider a penalised aggregate: assign failures the trajectory-length error,
  or report the fraction of sequence completed before failure, so a failure
  costs something.
* State the failure criterion explicitly (ATE above a threshold? tracking-lost
  flag? no output for $>N$ frames?). Different criteria move the numbers.

### 5.2 Sequence weighting

Averaging per-sequence ATEs weights a 30-second sequence identically with a
10-minute one. Pooling all poses weights by duration. Pooling all *segments*
(the KITTI $t_{rel}$ approach, see
[trajectory analysis §5](trajectory_analysis.ipynb)) weights by distance
travelled. These give **different rankings**, and none is wrong — but which one
you used has to be stated.

### 5.3 Mean vs median across sequences

Same argument as [trajectory analysis §6](trajectory_analysis.ipynb), one level
up: one catastrophic sequence dominates a mean across sequences. Report both,
plus max.

### 5.4 Normalisation

ATE in metres is not comparable between a 5 m indoor drone flight and a 5 km
drive. Report ATE **and** a length-normalised drift ($t_{rel}$ in %), or
restrict comparisons to a single scale regime.


## 6. Cherry-picking and leaderboard overfitting

Everything above is about noise. This section is about bias, which is larger.

**Per-sequence tuning.** Tuning parameters per sequence and reporting the
per-sequence best is fitting the test set. It is legitimate *if declared* (some
papers legitimately study parameter sensitivity), and it is misleading
otherwise. State plainly: *"one parameter set, fixed across all sequences"* — or
admit which parameters varied and how they were chosen.

**Sequence selection.** Reporting on the 5 of 11 EuRoC sequences where the
method wins. The fix is to report all sequences in the standard split and
discuss the losses.

**The test set as a dev set.** With a public leaderboard and unlimited
submissions, the test set becomes a validation set and the leaderboard ranking
becomes an estimate of *fit to the leaderboard*. The effect compounds across the
whole community, not just one author — every published method has already been
selected for performing well on EuRoC and KITTI, which is why relative rankings
so often fail to transfer to a new dataset. Report the number of submissions
where the platform tracks it, and treat a held-out dataset as the real test.

**Baseline asymmetry.** The most common bias of all: your method gets weeks of
tuning, the baseline gets its default config. If you did not tune the baseline
comparably, say so — it changes how the margin should be read.

**Version and configuration drift.** "OpenVINS 0.295 m" is not a fact about
OpenVINS; it is a fact about a commit, a config file, and a calibration. Pin all
three. This repo's own
[head-to-head comparison](../../vio_benchmark/docs/COMPARISON.md) and
[parameter reference](../../vio_benchmark/docs/PARAMETERS.md) exist for exactly
this reason.


## 7. Reporting checklist

Attach to any comparison table:

**Runs and variance**

1. Number of runs per (algorithm, sequence), and the seed policy.
2. Median and IQR (or mean and sd) — never a single run without the spread.
3. Whether the reported number is a mean, median, or best. If best, say so.

**Statistics**

4. Paired analysis across sequences, with the effect size and its CI, not just a
   $p$-value.
5. If bootstrapping, the resampling unit (sequence, not frame) and $B$.

**Failures**

6. Completion rate as its own column, and the failure criterion.
7. Whether averages exclude failed runs.

**Aggregation**

8. Weighting: per-sequence, per-pose, or per-segment.
9. Normalisation: raw ATE, $t_{rel}$, or both.

**Configuration**

10. One parameter set for all sequences — or which varied, and how they were chosen.
11. Commit hashes for every system compared, and the calibration used.
12. All sequences in the standard split, wins and losses.

**Everything from the metric layer** — alignment type, timestamp matching,
trimming, excluded frames — as listed in
[trajectory analysis §6](trajectory_analysis.ipynb).

## 8. References

* J. Demšar, *Statistical comparisons of classifiers over multiple data sets*,
  JMLR 7, 2006 — the canonical treatment of paired comparison over a set of
  benchmarks; the Wilcoxon signed-rank and Friedman tests it recommends are the
  nonparametric versions of §3.
* X. Bouthillier et al., *Accounting for variance in machine learning
  benchmarks*, MLSys 2021 — quantifies how much of a reported improvement is
  seed noise.
* P. Henderson et al., *Deep reinforcement learning that matters*, AAAI 2018 —
  the min-of-$N$ and cherry-picking failure modes of §1.1 and §6, measured.
* B. Efron, R. Tibshirani, *An Introduction to the Bootstrap*, 1993 — §4.
* Z. Zhang, D. Scaramuzza, *A tutorial on quantitative trajectory evaluation for
  visual(-inertial) odometry*, IROS 2018 — and the multi-run batch mode of
  `rpg_trajectory_evaluation`.
* J. Sturm et al., *A benchmark for the evaluation of RGB-D SLAM systems*, IROS
  2012 — the TUM RGB-D evaluation protocol.

## 9. See also

* [Trajectory analysis](trajectory_analysis.ipynb) — the metrics
* [Evaluation metrics for poses and trajectories](../vo_evaluation_metrics.ipynb) — ATE/RPE vs mAA
* [Estimator consistency](estimator_consistency.ipynb) — NEES/NIS, and where §2's runs are consumed
* [Runtime evaluation](runtime_evaluation.ipynb) — latency, throughput, tails
* [Head-to-head VIO comparison](../../vio_benchmark/docs/COMPARISON.md) · [Parameter reference](../../vio_benchmark/docs/PARAMETERS.md) · [Dataset recipes](../../vio_benchmark/docs/DATASETS.md)
